# Supervised Tabular Training

This notebook trains a compact flat classifier on the bundled Wine dataset. It follows the same API as Hello World, but uses a few more numeric inputs and exposes root embeddings during the same run. Use it as a tabular comparison point, not as the main nested-data story.

Import the runtime pieces used in the full training loop: Lightning for optimization and Polars for reading the bundled JSONL records.


In [1]:
import lightning.pytorch as lit
import polars as pl
import torch
from loguru import logger
from rich.pretty import pprint

import json2vec as j2v

logger.remove()

The Wine buffer is flat, but it gives enough numeric variation to make a clearer supervised example than handmade records. The model will use four chemistry fields to predict the cultivar.


In [2]:
records = pl.read_ndjson("docs/data/wine.jsonl").head(48)

records.head()

alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280_od315_of_diluted_wines,proline,cultivar
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
14.23,1.71,2.43,15.6,127.0,2.8,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,"""class_0"""
12.37,0.94,1.36,10.6,88.0,1.98,0.57,0.28,0.42,1.95,1.05,1.82,520.0,"""class_1"""
12.86,1.35,2.32,18.0,122.0,1.51,1.25,0.21,0.94,4.1,0.76,1.29,630.0,"""class_2"""
13.2,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.4,1050.0,"""class_0"""
12.33,1.1,2.28,16.0,101.0,2.05,1.09,0.63,0.41,3.27,1.25,1.67,680.0,"""class_1"""


The schema is the architecture. Four `Number` requests feed the root encoder, `cultivar` is the categorical target, and `embed=True` asks the root node to return embeddings after training.


In [3]:
model = j2v.Model.from_schema(
    j2v.Number("alcohol"),
    j2v.Number("malic_acid"),
    j2v.Number("color_intensity"),
    j2v.Number("proline"),
    j2v.Category("cultivar", target=True, max_vocab_size=4, topk=[2]),
    d_model=16,
    n_layers=1,
    n_heads=4,
    batch_size=8,
    embed=True,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

`PolarsDataModule(...)` reads the schema configuration from the model, so batch size, queries, targets, and tensorfield behavior stay tied to one object.

In [4]:
datamodule = j2v.PolarsDataModule(
    model,
    train=records,
    validate=records,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    observation_buffer_size=32,
    chunk_batch_size=32,
    sample_rate=1.0,
)

The tutorial trains for one tiny pass. In a real experiment this is where you would scale epochs, validation splits, callbacks, logging, and checkpointing.

In [5]:
trainer = lit.Trainer(
    accelerator="cpu",
    max_epochs=1,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False,
    enable_checkpointing=False,
    limit_train_batches=1,
    limit_val_batches=1,
)

trainer.fit(model=model, datamodule=datamodule)

GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer(limit_train_batches=1)` was configured so 1 batch per epoch will be used.


`Trainer(limit_val_batches=1)` was configured so 1 batch will be used.


/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
`Trainer.fit` stopped: `max_epochs=1` reached.


The plot confirms that the target and root embedding are both configured before inference.

In [6]:
model.plot()

Schema
record [array] embed
record
d_model=16  attention=mha  max_length=1  n_outputs=1  n_layers=1  n_heads=4  batch_size=8  parameters=26,039  arrays=1  
fields=5  targets=1  embeds=1  embed=True  n_linear=1
├── alcohol [number] 3,943 params
│   record/alcohol
│   query=[*].alcohol  pooling=query  objective=mae  weight=1.0  n_heads=4  n_linear=1  jitter=0.0  n_bands=8  offset=4
├── malic_acid [number] 3,943 params
│   record/malic_acid
│   query=[*].malic_acid  pooling=query  objective=mae  weight=1.0  n_heads=4  n_linear=1  jitter=0.0  n_bands=8  
│   offset=4
├── color_intensity [number] 3,943 params
│   record/color_intensity
│   query=[*].color_intensity  pooling=query  objective=mae  weight=1.0  n_heads=4  n_linear=1  jitter=0.0  n_bands=8  
│   offset=4
├── proline [number] 3,943 params
│   record/proline
│   query=[*].proline  pooling=query  objective=mae  weight=1.0  n_heads=4  n_linear=1  jitter=0.0  n_bands=8  offset=4
└── cultivar [category] target 3,659 params
    record/cultivar
    query=[*].cultivar  pooling=query  max_vocab_size=4  topk=[2]  weight=1.0  n_heads=4  p_prune=1.0  n_linear=1  
    n_bands=8  p_unavailable=0.01

After training, `predict` returns typed outputs for supervised targets and `embed` returns vectors from the nodes configured with `embed=True`.

In [7]:
batch = [[record] for record in records.to_dicts()[:3]]
pprint(model.predict(batch))
pprint(model.embed(batch))

{
│   'record/cultivar': {
│   │   'state': {
│   │   │   'valued': [0.29845482110977173, 0.27564507722854614, 0.28272977471351624],
│   │   │   'null': [0.41219791769981384, 0.4289396107196808, 0.41972556710243225],
│   │   │   'padded': [0.0393875353038311, 0.03640685975551605, 0.0376928448677063],
│   │   │   'masked': [0.17558565735816956, 0.18324404954910278, 0.18295493721961975],
│   │   │   'other': [0.07437402009963989, 0.07576436549425125, 0.0768967717885971]
│   │   },
│   │   'content': {
│   │   │   'value': ['class_1', 'class_1', 'class_1'],
│   │   │   'probability': [0.4173918664455414, 0.42106378078460693, 0.4143033027648926],
│   │   │   'topk': [
│   │   │   │   [
│   │   │   │   │   {'label': 'class_1', 'probability': 0.4173918664455414},
│   │   │   │   │   {'label': 'class_0', 'probability': 0.404940128326416}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'class_1', 'probability': 0.42106378078460693},
│   │   │   │   │   {'label': 'class_0', 'probability': 0.4010971486568451}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'class_1', 'probability': 0.4143033027648926},
│   │   │   │   │   {'label': 'class_0', 'probability': 0.4031635522842407}
│   │   │   │   ]
│   │   │   ]
│   │   }
│   }
}

{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   0.46143361926078796,
│   │   │   │   -0.03673524037003517,
│   │   │   │   0.07058490812778473,
│   │   │   │   -0.31384557485580444,
│   │   │   │   -0.20394162833690643,
│   │   │   │   -0.2223537415266037,
│   │   │   │   -0.17857085168361664,
│   │   │   │   0.418965220451355,
│   │   │   │   -0.01084270142018795,
│   │   │   │   0.08007039874792099,
│   │   │   │   0.16776934266090393,
│   │   │   │   0.0010131229646503925,
│   │   │   │   -0.5066830515861511,
│   │   │   │   0.16335095465183258,
│   │   │   │   0.23190221190452576,
│   │   │   │   -0.10922008752822876
│   │   │   ],
│   │   │   [
│   │   │   │   0.4681910574436188,
│   │   │   │   -0.012462046928703785,
│   │   │   │   0.14653849601745605,
│   │   │   │   -0.32285693287849426,
│   │   │   │   -0.23732949793338776,
│   │   │   │   -0.14080257713794708,
│   │   │   │   -0.18833668529987335,
│   │   │   │   0.3839087188243866,
│   │   │   │   -0.0712711289525032,
│   │   │   │   0.04467170313000679,
│   │   │   │   0.2603605091571808,
│   │   │   │   -0.059341080486774445,
│   │   │   │   -0.5047179460525513,
│   │   │   │   0.16951341927051544,
│   │   │   │   0.16223321855068207,
│   │   │   │   -0.08800902217626572
│   │   │   ],
│   │   │   [
│   │   │   │   0.4571879208087921,
│   │   │   │   0.03937191143631935,
│   │   │   │   0.14724569022655487,
│   │   │   │   -0.37444695830345154,
│   │   │   │   -0.21347901225090027,
│   │   │   │   -0.15412499010562897,
│   │   │   │   -0.24429206550121307,
│   │   │   │   0.37868380546569824,
│   │   │   │   -0.060578495264053345,
│   │   │   │   0.06732286512851715,
│   │   │   │   0.25034165382385254,
│   │   │   │   -0.0457703098654747,
│   │   │   │   -0.47923287749290466,
│   │   │   │   0.1498279869556427,
│   │   │   │   0.15971356630325317,
│   │   │   │   -0.06738266348838806
│   │   │   ]
│   │   ]
│   }
}